In [5]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler


# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

df = pd.read_csv('./output_file.csv')

# Feature engineering
df['w1w2'] = df['w1'] * df['w2']
df['w2w3'] = df['w2'] * df['w3']
df['w1w3'] = df['w1'] * df['w3']
df['w_sum'] = df['w1'] + df['w2'] + df['w3']
df['w_mean'] = df['w_sum'] / 3
df['w_std'] = df.apply(lambda x: np.std([x['w1'], x['w2'], x['w3']]), axis=1)

# Separate features and target
feature_columns = ['w1', 'w2', 'w3', 'wPL', 'w1w2', 'w2w3', 'w1w3', 'w_sum', 'w_mean', 'w_std']
X = df[feature_columns].values
y = df['range'].values

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Use MinMaxScaler instead of StandardScaler for better handling of the small dataset
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

# Create the model with regularization
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(len(feature_columns),),
                            kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu',
                            kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(8, activation='relu',
                            kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(1)
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                 loss='huber',  # Huber loss is more robust to outliers
                 metrics=['mae'])
    return model

# Create and train the model
model = create_model()

# Early stopping callback with increased patience
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    min_delta=1e-4
)

# Learning rate reduction callback
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=20,
    min_lr=1e-6
)

# Train the model
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=1000,
    batch_size=2,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Function to make predictions
def predict_range(w1, w2, w3, wPL):
    # Calculate engineered features
    w1w2 = w1 * w2
    w2w3 = w2 * w3
    w1w3 = w1 * w3
    w_sum = w1 + w2 + w3
    w_mean = w_sum / 3
    w_std = np.std([w1, w2, w3])
    
    # Prepare input data
    input_data = np.array([[w1, w2, w3, wPL, w1w2, w2w3, w1w3, w_sum, w_mean, w_std]])
    
    # Scale input data
    input_scaled = scaler_X.transform(input_data)
    
    # Make prediction
    prediction_scaled = model.predict(input_scaled)
    
    # Inverse transform the prediction
    prediction = scaler_y.inverse_transform(prediction_scaled.reshape(-1, 1))
    
    return prediction[0][0]

# Evaluate the model
train_predictions = scaler_y.inverse_transform(model.predict(X_train_scaled).reshape(-1, 1))
test_predictions = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1))

train_mse = np.mean((train_predictions.ravel() - y_train) ** 2)
test_mse = np.mean((test_predictions.ravel() - y_test) ** 2)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

print(f"Training RMSE: {train_rmse:.2f}")
print(f"Testing RMSE: {test_rmse:.2f}")

# Print detailed predictions for all data points
print("\nDetailed predictions for all data points:")
for i in range(len(X)):
    actual = y[i]
    predicted = predict_range(X[i][0], X[i][1], X[i][2], X[i][3])
    error = abs(actual - predicted)
    print(f"Row {i+1}:")
    print(f"Input (w1, w2, w3, wPL): {X[i][0]:.2f}, {X[i][1]:.2f}, {X[i][2]:.2f}, {X[i][3]:.6f}")
    print(f"Actual range: {actual:.2f}")
    print(f"Predicted range: {predicted:.2f}")
    print(f"Absolute error: {error:.2f}\n")

224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 763us/step - loss: 0.3405 - mae: 0.1544 - val_loss: 0.3569 - val_mae: 0.2115 - learning_rate: 5.0000e-04
Epoch 13/1000
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 758us/step - loss: 0.3330 - mae: 0.1580 - val_loss: 0.3379 - val_mae: 0.1832 - learning_rate: 5.0000e-04
Epoch 14/1000
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 777us/step - loss: 0.3234 - mae: 0.1477 - val_loss: 0.3270 - val_mae: 0.1770 - learning_rate: 5.0000e-04
Epoch 15/1000
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 776us/step - loss: 0.3151 - mae: 0.1471 - val_loss: 0.3127 - val_mae: 0.1604 - learning_rate: 5.0000e-04
Epoch 16/1000
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 761us/step - loss: 0.3050 - mae: 0.1429 - val_loss: 0.3009 - val_mae: 0.1518 - learning_rate: 5.0000e-04
Epoch 17/1000
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 753us/step - loss: 0.2967 - mae: 0.1485 - val_loss: 0.2906 - val_mae: 0.1501 - learning_rate: 5.0000e-04
Epoch 18/1000
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 761us/step - loss: 0.2851 - mae: 0.1346 - val_loss: 0.2796 - v